# __PROJECT_NAME__

一个项目 = 一个目录 = **一个 notebook** + 一个 `hooks.py`。

| 文件 | 谁写 | 作用 |
|---|---|---|
| `__PROJECT_NAME__.ipynb` | 你 + agent | 第 0 格固定不动，第 1 格起是你的业务 |
| `hooks.py` | agent | 这类项目特有的安装/打补丁/数据后处理 |
| `README.md` | 你 | 这个项目要干什么 |

配置在仓库根的 `config/__PROJECT_NAME__.toml`。

## 第 0 格 · 环境（固定块，永远不要改）

In [ ]:
# ============ 第 0 格：环境（固定块，永远不要改）============
# 唯一作用：把这台 VM 变成"能干活"的状态。幂等，随便重跑。
import os, subprocess, sys
from pathlib import Path

repo = Path("/content/colab-workbench")
if not repo.exists():
    subprocess.run(["git", "clone", "-q",
        "https://github.com/Elisabethhui/colab-workbench.git",
        str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=False)

# 同一个内核里重跑时，必须丢掉旧的模块缓存，否则改动不生效
for _m in [m for m in sys.modules if m == "bootstrap" or m.startswith("projects")]:
    del sys.modules[_m]
sys.path.insert(0, str(repo))

from bootstrap import setup
ctx = setup("__PROJECT_NAME__")   # ← 只改这一行：= config/__PROJECT_NAME__.toml

## 业务区（从这里开始是你的）

往下加你自己的格。产物写到 `ctx.paths.outputs`，日志写到 `ctx.paths.logs`——
它们在 Drive 上，换机器还在。

In [ ]:
print("数据目录:", ctx.paths.datasets)
print("产物目录:", ctx.paths.outputs)

In [ ]:
# 长任务记得边跑边往 Drive 写日志，断开也不丢
import time
from pathlib import Path

log = Path(ctx.paths.logs) / f"run_{time.strftime('%Y%m%d_%H%M%S')}.log"
with log.open("a") as fh:
    for step in range(3):
        fh.write(f"step {step}\n"); fh.flush()
        time.sleep(0.2)
print("日志已写入 Drive:", log)